# অধ্যায় ৭: চ্যাট অ্যাপ্লিকেশন নির্মাণ
## OpenAI API দ্রুত শুরু

এই নোটবুকটি [Azure OpenAI নমুনা রেপোজিটরি](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) থেকে অভিযোজিত, যা নোটবুক অন্তর্ভুক্ত করে যা [Azure OpenAI](notebook-azure-openai.ipynb) পরিষেবাগুলিতে অ্যাক্সেস করে।

পাইথন OpenAI API কিছু পরিবর্তনের সাথে Azure OpenAI মডেলগুলোর সাথেও কাজ করে। পার্থক্যগুলি সম্পর্কে আরও জানুন: [কিভাবে পাইথন দিয়ে OpenAI এবং Azure OpenAI এন্ডপয়েন্টের মধ্যে সুইচ করবেন](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# ওভারভিউ  
"বড় ভাষার মডেলগুলি এমন ফাংশন যা টেক্সট থেকে টেক্সটে ম্যাপ করে। একটি ইনপুট টেক্সট স্ট্রিং দেওয়া হলে, একটি বড় ভাষার মডেল পরবর্তী আসা টেক্সটটি পূর্বাভাস দেয়"(1)। এই "কুইকস্টার্ট" নোটবুকটি ব্যবহারকারীদের উচ্চ-স্তরের LLM ধারণা, AML দিয়ে শুরু করার জন্য মূল প্যাকেজের প্রয়োজনীয়তা, প্রম্পট ডিজাইনের একটি নরম পরিচিতি এবং বিভিন্ন ব্যবহার ক্ষেত্রে কয়েকটি সংক্ষিপ্ত উদাহরণ উপস্থাপন করবে। 


## বিষয়বস্তু তালিকা  

[সংক্ষিপ্ত বিবরণ](#overview)  
[কিভাবে OpenAI সার্ভিস ব্যবহার করবেন](#how-to-use-openai-service)  
[১. আপনার OpenAI সার্ভিস তৈরি করা](#1.-creating-your-openai-service)  
[২. ইনস্টলেশন](#2.-installation)    
[৩. পরিচয়পত্র](#3.-credentials)  

[ব্যবহার ক্ষেত্রসমূহ](#use-cases)    
[১. টেক্সট সারাংশ তৈরী করা](#1.-summarize-text)  
[২. টেক্সট শ্রেণীবদ্ধকরণ](#2.-classify-text)  
[৩. নতুন প্রোডাক্ট নাম তৈরি করা](#3.-generate-new-product-names)  
[৪. একটি শ্রেণীবিন্যাসকারী ফাইন টিউন করা](#4.fine-tune-a-classifier)  

[তথ্যসূত্র](#references)


### আপনার প্রথম প্রম্পট তৈরি করুন  
এই সংক্ষিপ্ত অনুশীলনটি একটি সাধারণ কাজ "সারাংশ তৈরি" করার জন্য OpenAI মডেলের কাছে প্রম্পট জমা দেওয়ার একটি মৌলিক পরিচয় প্রদান করবে।  


**ধাপসমূহ**:  
1. আপনার পাইথন পরিবেশে OpenAI লাইব্রেরি ইনস্টল করুন  
2. স্ট্যান্ডার্ড হেল্পার লাইব্রেরি লোড করুন এবং আপনি যে OpenAI সার্ভিস তৈরি করেছেন তার জন্য সাধারণ OpenAI সিকিউরিটি ক্রেডেনশিয়াল সেট করুন  
3. আপনার কাজের জন্য একটি মডেল নির্বাচন করুন  
4. মডেলের জন্য একটি সহজ প্রম্পট তৈরি করুন  
5. আপনার অনুরোধটি মডেল API-তে জমা দিন!  


### 1. OpenAI ইনস্টল করুন


In [ ]:
%pip install openai python-dotenv

### ২. সহায়ক লাইব্রেরি আমদানি করুন এবং শংসাপত্র তৈরি করুন


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### ৩. সঠিক মডেল খোঁজা  
GPT-4o এবং GPT-4o মিনি-এর মতো মডেলগুলি প্রাকৃতিক ভাষা বুঝতে এবং তৈরি করতে পারে, এবং বিভিন্ন কাজের জন্য উপযুক্ত বিভিন্ন ক্ষমতা এবং গতির সাথে OpenAI প্ল্যাটফর্মে উপলব্ধ।  


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## ৪. প্রম্পট ডিজাইন  

"বড় ভাষা মডেলগুলোর জাদু হচ্ছে, বিশাল পরিমাণ টেক্সটের ওপর এই পূর্বাভাস ত্রুটি সীমিত করার জন্য প্রশিক্ষিত হওয়ার মাধ্যমে, মডেলগুলো এমন ধারণা শেখে যা এই পূর্বাভাসের জন্য দরকার। উদাহরণস্বরূপ, তারা নিচের ধরনের ধারণা শিখে"(1):

* কিভাবে বানান করতে হয়
* কিভাবে ব্যাকরণ কাজ করে
* কিভাবে প্যারাফ্রেজ করতে হয়
* কিভাবে প্রশ্নের উত্তর দিতে হয়
* কিভাবে কথোপকথন চালাতে হয়
* অনেক ভাষায় কিভাবে লিখতে হয়
* কিভাবে কোড লিখতে হয়
* ইত্যাদি।

#### কিভাবে একটি বড় ভাষা মডেল নিয়ন্ত্রণ করবেন  
"বড় ভাষা মডেলের সব ইনপুটের মধ্যে সবচেয়ে প্রভাবশালী হল টেক্সট প্রম্পট(1)।

বড় ভাষা মডেলকে আউটপুট উৎপাদনের জন্য কয়েকটি উপায়ে প্রম্পট করা যেতে পারে:

নির্দেশনা: মডেলকে আপনি কী চান তা বলুন
সম্পূর্ণকরণ: মডেলকে আপনি যা চান তার শুরু শেষ করতে উৎসাহিত করুন
প্রদর্শনী: মডেলকে আপনি কী চান দেখান, যার মধ্যে হতে পারে:
প্রম্পটে কয়েকটি উদাহরণ
সূক্ষ্ম-টিউনিং প্রশিক্ষণ ডেটাসেটে শত শত বা হাজার হাজার উদাহরণ"



#### প্রম্পট তৈরি করার তিনটি মৌলিক নির্দেশিকা রয়েছে:

**দেখান এবং বলুন**। যা চান তা স্পষ্ট করুন, হয় নির্দেশনা দিয়ে, উদাহরণ দিয়ে, অথবা উভয়ের সমন্বয়ে। যদি আপনি মডেলকে একটি তালিকা বর্ণানুক্রমিক ক্রমে সাজাতে চান অথবা একটি প্যারাগ্রাফকে অনুভূতি অনুযায়ী শ্রেণীবদ্ধ করতে চান, দেখান যে আপনি সেটাই চান।

**গুণগতমান সম্পন্ন তথ্য দিন**। আপনি যদি একটি শ্রেণীবিশেষক তৈরি করতে চান বা মডেলকে কোনো প্যাটার্ন অনুসরণ করতে চান, তখন নিশ্চিত করুন যথেষ্ট উদাহরণ রয়েছে। আপনার উদাহরণগুলি অবশ্যই পুনরায় পরীক্ষা করুন — মডেল সাধারণ বানানের ভুল দেখতে পেয়ে সাধারণত উত্তর দিতে পারে, কিন্তু কখনও কখনও ধরে নিতে পারে এটা ইচ্ছাকৃত এবং এটি উত্তরকে প্রভাবিত করতে পারে।

**আপনার সেটিংস পরীক্ষা করুন।** তাপমাত্রা এবং top_p সেটিংস মডেল কতটা নিশ্চয়তার সাথে উত্তর তৈরি করে তা নিয়ন্ত্রণ করে। আপনি যদি এমন উত্তর চান যার একমাত্র সঠিক উত্তর আছে, তাহলে এগুলো কম রাখবেন। আপনি যদি আরও বৈচিত্র্যময় উত্তর চান, তাহলে এগুলো বেশি রাখতে পারেন। সবচেয়ে বড় ভুল হচ্ছে এই সেটিংগুলোকে "বুদ্ধিমত্তা" বা "সৃজনশীলতা" নিয়ন্ত্রণ হিসেবে ধরা।


উৎস: https://learn.microsoft.com/azure/ai-services/openai/overview


### ৫. জমা দিন!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### একই কলটি পুনরাবৃত্তি করুন, ফলাফলগুলি কীভাবে তুলনা করে?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## টেক্সট সংক্ষেপ  
#### চ্যালেঞ্জ  
একটি টেক্সট অংশের শেষে 'tl;dr:' যোগ করে টেক্সট সংক্ষেপ করুন। লক্ষ্য করুন কীভাবে মডেল অতিরিক্ত নির্দেশনা ছাড়াই অনেকগুলো কাজ কিভাবে সম্পাদন করতে হয় তা বুঝতে পারে। আপনি মডেলের আচরণ পরিবর্তন করতে এবং প্রাপ্ত সংক্ষেপকরণ কাস্টমাইজ করতে tl;dr এর চেয়ে আরও বর্ণনামূলক প্রম্পট নিয়ে পরীক্ষা-নিরীক্ষা করতে পারেন(3)।  

সাম্প্রতিক কাজগুলি একটি বড় টেক্সট কর্পাসে প্রি-ট্রেইনিং করে এবং তারপর নির্দিষ্ট একটি কাজের উপর ফাইন-টিউনিং করে অনেক NLP কাজ এবং বেঞ্চমার্কগুলিতে উল্লেখযোগ্য উন্নতি দেখিয়েছে। যদিও সাধারণত স্থাপত্যে কাজ-অজ্ঞাত (task-agnostic), এই পদ্ধতিতে এখনও হাজার হাজার বা দশ হাজারেরও বেশি উদাহরণের টাস্ক-নির্দিষ্ট ফাইন-টিউনিং ডেটাসেট প্রয়োজন। বিপরীতে, মানুষ সাধারণত শুধু কয়েকটি উদাহরণ বা সহজ নির্দেশনা থেকে একটি নতুন ভাষা কাজ সম্পাদন করতে পারে—একটি বিষয় যা বর্তমান NLP সিস্টেমগুলি এখনও বড় ধরনের সংগ্রাম করছে। এখানে আমরা দেখাচ্ছি ভাষার মডেলগুলি বড় পরিসরে ব্যবহার করলে কাজ-অজ্ঞাত, কয়েক-শট পারফরম্যান্স অনেক উন্নত হয়, কখনও কখনও আগের শ্রেষ্ঠ ফাইন-টিউনিং পদ্ধতির সাথে প্রতিযোগিতামূলক পর্যায়েও পৌঁছে।  



Tl;dr


# বিভিন্ন ব্যবহারিক ক্ষেত্রে অনুশীলন  
1. টেক্সট সংক্ষিপ্তসার  
2. টেক্সট শ্রেণীবিভাজন  
3. নতুন পণ্যের নাম তৈরি  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## টেক্সট শ্রেণীবিভাজন  
#### চ্যালেঞ্জ  
ইন্টারেন্স টাইমে সরবরাহ করা ক্যাটাগরিতে আইটেমগুলি শ্রেণীবদ্ধ করুন। নিচের উদাহরণে, আমরা শ্রেণীবিভাজনের জন্য উভয়ই ক্যাটাগরি এবং টেক্সট প্রম্পটে(*playground_reference) প্রদান করি। 

গ্রাহক অনুসন্ধান: হ্যালো, আমার ল্যাপটপের কীবোর্ডের একটি কী সম্প্রতি ভেঙে গেছে এবং আমাকে একটি প্রতিস্থাপন নিতে হবে:

শ্রেণীবদ্ধকৃত বিভাগ:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## নতুন পণ্য নাম তৈরি করুন
#### চ্যালেঞ্জ
উদাহরণ শব্দ থেকে পণ্য নাম তৈরি করুন। এখানে আমরা যে পণ্যটির জন্য নাম তৈরি করতে যাচ্ছি তার তথ্য প্রম্পটে অন্তর্ভুক্ত করেছি। আমরা একই ধরনের একটি উদাহরণও প্রদান করেছি যাতে আমরা যে প্যাটার্ন পেতে চাই তা বোঝানো যায়। আমরা এলোমেলোতা এবং আরও উদ্ভাবনী প্রতিক্রিয়া বৃদ্ধির জন্য তাপমাত্রার মানও বেশি সেট করেছি।

পণ্যের বিবরণ: একটি হোম মিল্কশেক মেকার
বীজ শব্দ: দ্রুত, স্বাস্থ্যকর, কম্প্যাক্ট।
পণ্য নাম: HomeShaker, Fit Shaker, QuickShake, Shake Maker

পণ্যের বিবরণ: একটি জুতা যা যেকোন পায়ের আকারে ফিট হতে পারে।
বীজ শব্দ: অভিযোজযোগ্য, উপযুক্ত, ওমনি-ফিট।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# রেফারেন্স  
- [Openai কুকবুক](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry পোর্টাল](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [টেক্সট শ্রেণিবদ্ধ করতে GPT মডেল ফাইন-টিউনিং এর জন্য সেরা অভ্যাসসমূহ](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# আরও সাহায্যের জন্য  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# অবদানকারীরা
* লুইস লি  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
